# app

## bootstrap

In [1]:
import os
from pathlib import Path

def is_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False
    

IN_COLAB = is_colab()

if IN_COLAB:
    REPO_ROOT = Path("/content") / "BrazingSense"
else:
    REPO_ROOT = Path.cwd().resolve()

if not (REPO_ROOT / ".git").exists():
    raise RuntimeError(f"REPO_ROOT не похож на корень репозитория: {REPO_ROOT}")

os.chdir(REPO_ROOT)

## packages

## env

In [8]:
FPS = 10
ROOT = Path()
DRIVE_FOLDER = Path('/content/drive/MyDrive/Colab Notebooks/Diploma')
EXPERIMENT_NAME = "mobilenet_architecture_evolution"

In [3]:
DATA = ROOT / "data" 

DATA_ANNOTATIONS = DATA / "annotations" 
STAGE_INTERVALS_PATH = DATA_ANNOTATIONS / "stage_intervals.csv"
FRAME_LABELS_PATH = DATA_ANNOTATIONS / f"frame_labels_{FPS}.csv"
FRAME_STATS_PATH = DATA_ANNOTATIONS / f"frame_dataset_stats_{FPS}.json"

DATA_PROCESSED = DATA / 'processed'
DATA_PROCESSED_FRAMES = DATA_PROCESSED / f'frames_{FPS}'

DATA_RAW = DATA / 'raw'

DRIVE_DATA_RAW = DRIVE_FOLDER / DATA_RAW
DRIVE_PROCESSED_FRAMES_FOLDER = DRIVE_FOLDER / DATA_PROCESSED_FRAMES

In [4]:
REPORTS = ROOT / "reports" 

In [9]:
MODELS_DIR = ROOT / "models"
CHECKPOINTS_DIR = MODELS_DIR / "checkpoints" / EXPERIMENT_NAME

DRIVE_CHECKPOINTS_DIR = DRIVE_FOLDER / CHECKPOINTS_DIR

# lib

# runtime

## data

In [12]:
import subprocess

if IN_COLAB:
    from google.colab import drive # type: ignore
    drive.mount("/content/drive")

    copy_frames = ["cp", "-r", f"{DRIVE_PROCESSED_FRAMES_FOLDER}/.", f"{DATA_PROCESSED_FRAMES}/"]
    with open("copy_frames.log", "w") as frames_log:
        copy_frames_process = subprocess.Popen(copy_frames, stdout=frames_log, stderr=subprocess.STDOUT, text=True)

    copy_checkpoints = ["cp", "-r", f"{DRIVE_CHECKPOINTS_DIR}/.", f"{CHECKPOINTS_DIR}/"]
    with open("copy_checkpoints.log", "w") as checkpoints_log:
        copy_checkpoints_process = subprocess.Popen(copy_checkpoints, stdout=checkpoints_log, stderr=subprocess.STDOUT, text=True)

    copy_raw_data = ["cp", "-r", f"{DRIVE_DATA_RAW}/.", f"{DATA_RAW}/"]
    with open("copy_raw_data.log", "w") as data_log:
        copy_raw_data_process = subprocess.Popen(copy_raw_data, stdout=data_log, stderr=subprocess.STDOUT, text=True)

    # !cp -r -v "$DRIVE_PROCESSED_FRAMES_FOLDER"/. "$DATA_PROCESSED_FRAMES"/

Mounted at /content/drive


In [15]:
# print("copy_frames_process", copy_frames_process.poll())
print("copy_checkpoints_process", copy_checkpoints_process.poll())
print("copy_raw_data_process", copy_raw_data_process.poll())

copy_checkpoints_process None
copy_raw_data_process None


In [ ]:
# !tail copy_checkpoints.log

Process None


In [ ]:
from google.colab import drive # type: ignore
drive.mount("/content/drive")

!cp -r "$DRIVE_CHECKPOINTS_DIR"/. "$CHECKPOINTS_DIR"/
!cp -r "$DRIVE_DATA_RAW"/. "$DATA_RAW"/

In [ ]:
--model mobilenet_v3_small | resnet18 | mobilenet_v3_small_architecture_evo
--checkpoint models/checkpoints/final_neural_stage_classification_10/resnet18_10fps_balanced_best_10fps.pt 224 0.4
--checkpoint models/checkpoints/resnet_input_size_cpu/resnet18_64_finetuned_best.pt 64 0.2/0.4
--checkpoint models/checkpoints/mobilenet_evolutionary_search/mobilenet_evo_best_finetuned_best.pt 224 0.2
--checkpoint models/checkpoints/mobilenet_architecture_evolution/mobilenet_arch_evo_039_best.pt 224 0.2

In [ ]:
PYTHONPATH=src python3 scripts/run_neural_inference_video.py \
  --video data/raw/MVI_6266.MOV \
  --checkpoint models/checkpoints/mobilenet_architecture_evolution/mobilenet_arch_evo_039_best.pt \
  --output docs/final/MVI_6266_MobileNet_HS_CPU.mp4 \
  --model mobilenet_v3_small_architecture_evo \
  --roi 470,280,430,290 \
  --image-size 224 \
  --postprocess state_machine \
  --trigger-threshold 0.2 \
  --trigger-confirm-frames 1 \
  --device cpu

  # --enable-cv-trigger \
  # --cv-threshold 0.37 \
  # --cv-confirm-frames 2 \

In [ ]:
PYTHONPATH=src python3 scripts/benchmark_neural_latency.py \
  --video data/raw/MVI_6266.MOV \
  --checkpoint models/checkpoints/mobilenet_evolutionary_search/mobilenet_evo_best_finetuned_best.pt \
  --model mobilenet_v3_small \
  --roi 470,280,430,290 \
  --image-size 224 \
  --postprocess state_machine \
  --trigger-threshold 0.4 \
  --trigger-confirm-frames 1 \
  --device auto \
  --output reports/neural_latency_mobilenet_evo_cpu/MVI_6266_latency.csv \
  --summary-output reports/neural_latency_mobilenet_evo_cpu/MVI_6266_summary.json